## Day 3: Analyzing ERPs in a Go/NoGo Task  · Behavioral Data Analysis: Go/NoGo Task 
---
---
Welcome to the final part of our workshop! By now, our EEG data is fully preprocessed, cleaned from artifacts,we cut into epochs around our stimuli and analysed behavior.

Now comes the exciting part: We will extract the **Event-Related Potentials (ERPs)**. ERPs represent the brain's averaged electrical response to a specific event. In our Go/NoGo task, we are particularly interested in the differences between trials where participants had to act (Go) and trials where they had to inhibit their response (NoGo).

# Part A: Analyzing ERPs in a Go/NoGo Task 
### Setup and Loading Data
First, let's import the necessary libraries and load our preprocessed epochs and the electrode coordinates.

### How to use this notebook
- Replace every `???` with your own code. 
- **Master Solution** will be shown at the end of the day
- Read every markdown cell before writing code — it explains what to do and *why*
----

In [ ]:
# Interactive Plots
%matplotlib widget

import mne
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

mne.set_log_level('WARNING')  # suppress info messages, keep warnings

import os.path as op

# Load the preprocessed epochs
sub_id   = '???'
DATA_DIR = f'../data/hackathon_data/derivatives/{sub_id}/eeg'


epo_path = ???
epochs = ???

# Use standard electrode positions (same as Day 2)
montage = ???
epochs.set_montage(???)

print(epochs.info)

## Task 1: Calculating and Plotting Evoked Responses

To reveal the underlying ERP, we need to average our single trials (epochs). Random background noise (like spontaneous brain activity) will average out to zero, leaving only the signal that is time-locked to our stimuli.

**Your Task:**
1. Create an averaged response (`Evoked` object) for the **Go** trials.
2. Create an averaged response for the **NoGo** trials.
3. Plot the ERP for the NoGo condition.

In [ ]:
# Create Evoked objects by averaging epochs per condition
evoked_go = ???
evoked_nogo = ???

# Compute the difference wave (NoGo minus Go)
# This isolates the inhibition-related activity (N2, P3 NoGo effect)
evoked_diff = 
evoked_diff.comment = 'NoGo − Go'

# Plot the Evoked response for the NoGo condition
# Spatial_colors=True for aesthetics
fig_go   = ???
fig_nogo = ???(spatial_colors=True, titles='ERP: NoGo Condition')
fig_diff = ???

## Task 2: Comparing Conditions

Looking at conditions separately is nice, but comparing them in a single plot is much more insightful! We want to see if the brain's response differs when inhibiting an action compared to executing it.

**Your Task:**
Plot the **Go** and **NoGo** conditions together. Focus on a specific region of interest (e.g., the frontal electrode 'Fz', which is typical for inhibitory control tasks).

In [ ]:
# Defining a dictionary containing both conditions
evokeds_??? = {'Go': evoked_go, 'NoGo': evoked_nogo}

# Plot the comparison
fig_compare = mne.viz.???(
    ??? #as a hint: add couple of extra customization parameters for aesthetics, too
)

## Bonus - Group level 

In [ ]:
# Grand-Average ERPs across all participants 
participantsA = [???]

evoked_go_list   = []
evoked_nogo_list = []
evoked_diff_list = []

for sub_id in participantsA:
    DATA_DIR = f'../data/hackathon_data/derivatives/{sub_id}/eeg'
    epo_path = ???
    epo = ???

    if len(epo) == 0:
       ???

    ev_go   = ???
    ev_nogo = ???
    ev_diff = mne.combine_???

    evoked_go_list.append(???)
    evoked_nogo_list.append(???)
    evoked_diff_list.append(???)
    print(f'  {sub_id}: Go = {len(epo["Go"])} epochs, NoGo = {len(epo["NoGo"])} epochs')

# Compute grand averages
grand_go   = ???
grand_nogo = ???
grand_diff = ???

grand_go.comment   = 'Go'
grand_nogo.comment = 'NoGo'
grand_diff.comment = 'NoGo − Go'

# Plot comparison at Fz
fig = ???
)
print(f'\nIncluded in grand average: {len(evoked_go_list)} participants')

## Task 3: Statistical Testing 

Visualizing differences in ERPs is crucial, but we also need to prove that these differences are statistically significant. Is the brain's response in the NoGo condition truly different from the Go condition, or is it just random noise?

To keep things simple and intuitive, we will focus on a **Region of Interest (ROI)**. We will extract the mean voltage for a specific electrode in a specific time window for every single trial, and then compare the two groups of trials using an independent t-test.

**Your Task:**
1. Define an ROI: Select the frontal electrode `'Fz'` and a time window capturing the N2 component (e.g., `0.250` to `0.350` seconds).
2. Extract the data for this ROI from both the Go and NoGo epochs and average the voltage across the time window.
3. Perform an independent t-test (`scipy.stats.ttest_ind`) to compare the trial-by-trial mean amplitudes.

In [ ]:
# Define the Region of Interest (ROI)
channel_of_interest = ???
t_start = 0.250
t_end = 0.350

# Extract data for the ROI
# pick() selects the channel, crop() selects the time window, get_data() converts to numpy array
go_data = epochs['Go']???
nogo_data = epochs['NoGo'].???

# Average across the time window (the last dimension, axis=-1) to get one single voltage value per trial
go_means = np.???
nogo_means = np.???

# 3. Perform the Independent T-Test
t_stat, p_val = stats.??? # Welch's t-test

# Print the results
print(f"Statistical Analysis for {channel_of_interest} ({t_start*1000} - {t_end*1000} ms)")
print(f"Mean amplitude Go:   {np.mean(go_means)*1e6:.2f} µV")
print(f"Mean amplitude NoGo: {np.mean(nogo_means)*1e6:.2f} µV")
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value:     {p_val:.4f}")

if p_val < ???:
    print("Statistically significant!")
else:
    print("No significant difference.")

## Task 4: Spatial Distribution (Topographies)

A simple line plot shows us *when* something happens, but a **Topographical Map** shows us *where* on the scalp the activity is distributed. 

**Your Task:**
Create a topomap for the **NoGo** condition. Choose specific time points to plot, for instance, the typical time windows for the N2 and P3 components (e.g., 250 ms and 400 ms after the stimulus).

In [ ]:
# Define the time points of interest (in seconds)
times_of_interest = [0.250, 0.400]

# Plot the topographies
fig_topo_nogo = evoked_nogo.plot_???(
    ???
)
fig_topo_nogo.suptitle('NoGo Topographies (N2 & P3)')

fig_topo_diff = ???

## Bonus - Group level 

In [ ]:
# Grand-Average Topographies
# (uses grand_go, grand_nogo, grand_diff from the cell above)

times_of_interest = [0.250, 0.400]

fig_topo_go = ???
fig_topo_go.suptitle(f'Grand-Average Go (n = {len(evoked_go_list)})')

fig_topo_nogo = ???
fig_topo_nogo.suptitle(f'Grand-Average NoGo (n = {len(evoked_nogo_list)})')

fig_topo_diff = ???
fig_topo_diff.suptitle(f'Grand-Average NoGo − Go (n = {len(evoked_diff_list)})')

---
---
# Part B: Behavioral Data Analysis
Before we dive into the complex world of brainwaves (EEG), we need to look at the behavioral data.

In a Go/NoGo task, we generate a `.csv` file that logs every single trial. We are mainly interested in two behavioral metrics:
1. **Accuracy:** The percentage of correct responses.
2. **Reaction Time (RT):** How fast the participant pressed the button.

### Setup and Loading Data
You may import `pandas` for data handling and `seaborn`/`matplotlib` for some aesthetic plots.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Set style for our plots
sns.set_theme(???)


sub_id = '???'
# Load the behavioral data 
# (Make sure the path matches your .csv file)
CSVFILE_DIR = f'../data/hackathon_data/{sub_id}/beh'

# Load the behavioral data 
# (Make sure the path matches your .csv file)
csv_files = glob.glob(os.path.join(CSVFILE_DIR, '*.csv'))[0] #automatically matches the csv file in your path (we select only the first one, as there only is one anyways)

print(f"Found {os.path.basename(csv_files)}") #safety check
df_raw = ???

## Clean Data from PsychoPy Issues
PsychoPy writes a separate line in the CSV file for **each routine** (TrialIntro, Delay, Breaks, TrialOutro) – even if there is no trial data there. These lines have no value in the `type` column and must be filtered out.

In [ ]:
df = ???(subset=['type']).???()
df = df[df['type'].isin(['go', 'nogo'])].copy()

print(f"Left: {???(df)}")
print(f"Deleted: {len(df_raw) - len(df)} - lines that contaminate the data (Intro, Delay, Breaks, Outro)\n")


# Look for Number of Go vs. NoGo Trials (should be around 75/25)
print(???['type'].value_counts())
print(f"\nTotal: {len(df)} Trials")

# Check: for how many of the Trials do we have response times?
# the two df columns are identical (due to PsychoPy output format)
print("TrialResp.rt vorhanden:", df['TrialResp.rt'].???().sum())
print("TrialLoop.TrialResp.rt vorhanden:", df['TrialLoop.TrialResp.rt'].notna().sum())

# Check: how many of the Trials have been answered correctly?
# the two df columns are identical (due to PsychoPy output format)
print("TrialResp.corr vorhanden:", ???['TrialResp.corr'].notna().sum())
print("TrialLoop.TrialResp.corr vorhanden:", df['TrialLoop.TrialResp.corr'].???().???())

## Task 1: Accuracy (Hits vs. Correct Rejections) 

In our task, the participant had to press the spacebar for certain images (Go) and withhold their response for others (NoGo). 

Our dataset has a column called `TrialResp.corr` which is `1` if the participant did the right thing, and `0` if they made a mistake.

**Your Task:**
1. Group the dataset by condition (Go vs. NoGo). *Hint: You can figure out the condition by looking at the `corrAns` column!*
2. Calculate the mean accuracy (percentage) for both conditions.
3. Plot the results in a simple bar chart.

In [ ]:
# Define Go and NoGo trials based on the 'corrAns' column
# If the correct answer was 'None' or empty, it was a NoGo trial!
df['Condition'] = ???

# Calculate Accuracy (mean of the 'TrialResp.corr' column * 100)
### NOTE: Remove the groupby and plotting logic for the exercise
f['TrialResp.corr'] = df['???'].fillna(0.0)  
#important, since NAs will otherwise simply be omitted from the mean calculation, but we want to treat NAs as missed trials

accuracy = ???

# Plot Accuracy
# Create a bar plot showing task accuracy by condition
## TODO: Create barplot with x=accuracy.index, y=accuracy.values
## Hint: Use sns.barplot() and set a fitting color palette (e.g. #4C72B0 and #C44E52)
# TODO: add a title, y-axis label, and set proper y-axis limits
plt.figure(???)
#Hint: you need to add more code lines to create the barplot and customize it

print(f"Hit Rate (Correct Go): {accuracy['Go']:.2f}%")
print(f"Correct Rejection Rate (Correct NoGo): {accuracy['NoGo']:.2f}%")

## Task 2: Reaction Times 

Reaction times (RTs) tell us a lot about cognitive control. Often, when participants make a mistake on a NoGo trial (a **False Alarm**), it happens because they are reacting too fast (impusively), failing to inhibit their motor response in time.

**Your Task:**
1. Filter out all trials where no button was pressed (where RT is missing/NaN).
2. Separate the remaining trials into **Hits** (Correct Go responses) and **False Alarms** (Incorrect NoGo responses).
3. Create a violin plot or boxplot to compare the reaction times of Hits vs. False Alarms.

In [ ]:
# Filter out trials where no response was made (RT is NaN)
# Hint: Use .dropna()
df_responded = df.???

# Determine Response Type: Hit vs False Alarm
# NOTE: The logic below classifies responses. 
df_responded['ResponseType'] = df_responded.apply(
    lambda row: 'Hit' if row[???] == 'Go' and row[???] == 1
    else ('False Alarm' if row[???] == 'NoGo' and row[???] == 0 else 'Other'),
    axis=1
)

# Filter to only keep Hits and False Alarms
# Hint: Use .isin() to filter for the list ['Hit', 'False Alarm']
df_rts = ???

# Plot Reaction Times with a violin plot. Again, customize the plot with a title and axis labels.
plt.figure(figsize=???)
sns.???
plt.show()

# Quick statistical summary
print("Summary; Reaction Time (in seconds)")
# Hint: Group by 'ResponseType' and describe the 'TrialResp.rt' column
summary = df_rts.???
print(summary)